# Strong Multilabel Video Baseline

This notebook replaces the earlier CLIP experiment with a stronger and cleaner multilabel baseline:

- direct frame loading from `datasets/data-from-juniors/frames`
- uniform temporal sampling across the full clip
- ImageNet-pretrained 2D backbone with temporal attention pooling
- weighted `BCEWithLogitsLoss` for class imbalance
- AMP, gradient clipping, warmup-freeze then full finetuning
- per-label threshold tuning on the validation set
- checkpointed inference utilities

If `iterstrat` is installed, the split will use iterative multilabel stratification. Otherwise the notebook falls back to the best of many random splits.


In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision.models as tv_models
from torch.cuda.amp import GradScaler
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

warnings.filterwarnings("ignore", category=UserWarning)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

CFG = {
    "seed": 42,
    "image_size": 224,
    "num_frames": 8,
    "batch_size": 8,
    "num_workers": 4,
    "epochs": 10,
    "warmup_epochs": 1,
    "val_size": 0.20,
    "head_lr": 3e-4,
    "backbone_lr": 3e-5,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "patience": 4,
    "max_pos_weight": 20.0,
    "merge_rare_labels": True,
}


def resolve_data_root() -> Path:
    candidates = [
        Path.cwd() / "datasets" / "data-from-juniors",
        Path.cwd().parent / "datasets" / "data-from-juniors",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find datasets/data-from-juniors from the notebook working directory."
    )


DATA_ROOT = resolve_data_root()
FRAMES_ROOT = DATA_ROOT / "frames"
CSV_PATH = DATA_ROOT / "dataset.csv"
CHECKPOINT_PATH = Path.cwd() / "best_model_fast_baseline.pt"
CHECKPOINT_PATH_EMA = Path.cwd() / "best_model_fast_baseline_ema.pt"


print("device:", DEVICE)
print("data root:", DATA_ROOT)
print("checkpoint:", CHECKPOINT_PATH)
print("EMA checkpoint:", CHECKPOINT_PATH_EMA)


device: cuda
data root: D:\hvc\datasets\data-from-juniors
checkpoint: D:\hvc\best_model_fast_baseline.pt


In [9]:
df = pd.read_csv(CSV_PATH)

original_label_columns = [column for column in df.columns if column != "video_id"]
rare_labels = [
    "ethinity_hate",
    "caste_based_hate",
    "social_hate",
    "religion_hate",
    "controversial",
]

if CFG["merge_rare_labels"]:
    available_rare_labels = [column for column in rare_labels if column in df.columns]
    df["rare_hate"] = df[available_rare_labels].max(axis=1)
    label_columns = [
        column for column in original_label_columns if column not in available_rare_labels
    ] + ["rare_hate"]
    df = df[["video_id"] + label_columns]
else:
    label_columns = original_label_columns

df[label_columns] = df[label_columns].fillna(0).astype(int)
df["label_count"] = df[label_columns].sum(axis=1)
df = df[df["label_count"] > 0].copy()


def frame_dir_for(video_id: str) -> Path:
    return FRAMES_ROOT / Path(str(video_id)).stem


def count_frames(video_id: str) -> int:
    frame_dir = frame_dir_for(video_id)
    if not frame_dir.exists():
        return 0
    return sum(
        1
        for path in frame_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )


df["frame_count"] = df["video_id"].map(count_frames)
df = df[df["frame_count"] > 0].drop(columns=["label_count"]).reset_index(drop=True)

label_stats = pd.DataFrame(
    {
        "positives": df[label_columns].sum().astype(int),
        "prevalence_%": (df[label_columns].mean() * 100).round(2),
    }
).sort_values("positives", ascending=False)

print(f"samples with frames: {len(df):,}")
print(f"labels: {len(label_columns)}")
display(label_stats)
df.head()


samples with frames: 1,782
labels: 15


,positives,prevalence_%
humour,1353,75.93
sensitive,698,39.17
anger,373,20.93
derogatory__lang,351,19.70
emotional,164,9.20
threat,149,8.36
political_hate,131,7.35
gender_hate,109,6.12
rare_hate,78,4.38
indv_hate,68,3.82


,video_id,generic,humour,positive,sensitive,derogatory__lang,threat,sexuality_hate,nationality_hate,political_hate,informative,anger,emotional,indv_hate,gender_hate,rare_hate,frame_count
0,54,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,8
1,55,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,8
2,56,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,8
3,57,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,8
4,58,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,8


In [10]:
def score_split(y_full, y_train, y_val):
    full_prev = y_full.mean(axis=0)
    train_prev = y_train.mean(axis=0)
    val_prev = y_val.mean(axis=0)
    drift = np.abs(train_prev - full_prev).mean() + np.abs(val_prev - full_prev).mean()
    missing_train = int((y_train.sum(axis=0) == 0).sum())
    missing_val = int(((y_full.sum(axis=0) >= 2) & (y_val.sum(axis=0) == 0)).sum())
    return drift + missing_train * 1.0 + missing_val * 0.5


def multilabel_train_val_split(frame_df, label_cols, val_size=0.2, seed=42, tries=200):
    y = frame_df[label_cols].values
    all_indices = np.arange(len(frame_df))

    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=val_size,
            random_state=seed,
        )
        train_idx, val_idx = next(splitter.split(np.zeros(len(frame_df)), y))
        method = "iterstrat"
    except Exception:
        rng = np.random.default_rng(seed)
        best_pair = None
        best_score = float("inf")

        for _ in range(tries):
            random_state = int(rng.integers(0, 1_000_000_000))
            train_idx, val_idx = train_test_split(
                all_indices,
                test_size=val_size,
                random_state=random_state,
                shuffle=True,
            )
            current_score = score_split(y, y[train_idx], y[val_idx])
            if current_score < best_score:
                best_score = current_score
                best_pair = (train_idx, val_idx)

        train_idx, val_idx = best_pair
        method = f"best-of-{tries} random splits"

    train_df = frame_df.iloc[train_idx].reset_index(drop=True)
    val_df = frame_df.iloc[val_idx].reset_index(drop=True)
    return train_df, val_df, method


train_df, val_df, split_method = multilabel_train_val_split(
    df,
    label_columns,
    val_size=CFG["val_size"],
    seed=CFG["seed"],
)

split_summary = pd.DataFrame(
    {
        "train_pos": train_df[label_columns].sum().astype(int),
        "val_pos": val_df[label_columns].sum().astype(int),
        "train_prev_%": (train_df[label_columns].mean() * 100).round(2),
        "val_prev_%": (val_df[label_columns].mean() * 100).round(2),
    }
)

print("split method:", split_method)
print(f"train samples: {len(train_df):,} | val samples: {len(val_df):,}")
display(split_summary.sort_values("train_pos", ascending=False))


split method: best-of-200 random splits
train samples: 1,425 | val samples: 357


,train_pos,val_pos,train_prev_%,val_prev_%
humour,1083,270,76.00,75.63
sensitive,558,140,39.16,39.22
anger,298,75,20.91,21.01
derogatory__lang,280,71,19.65,19.89
emotional,136,28,9.54,7.84
threat,120,29,8.42,8.12
political_hate,104,27,7.30,7.56
gender_hate,88,21,6.18,5.88
rare_hate,61,17,4.28,4.76
indv_hate,54,14,3.79,3.92


In [11]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def numeric_frame_key(path: Path):
    return (0, int(path.stem)) if path.stem.isdigit() else (1, path.stem)


def list_frame_paths(frame_dir: Path):
    frame_paths = [
        path
        for path in frame_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    return sorted(frame_paths, key=numeric_frame_key)


def sample_frame_paths(frame_dir: Path, num_frames: int):
    frame_paths = list_frame_paths(frame_dir)
    if not frame_paths:
        raise FileNotFoundError(f"No frames found in {frame_dir}")
    if len(frame_paths) == 1:
        return frame_paths * num_frames
    indices = np.linspace(0, len(frame_paths) - 1, num=num_frames)
    indices = np.clip(np.round(indices).astype(int), 0, len(frame_paths) - 1)
    return [frame_paths[index] for index in indices]


class ClipTransform:
    def __init__(self, image_size: int = 224, train: bool = True):
        self.image_size = image_size
        self.train = train

    def __call__(self, images):
        processed = []

        if self.train:
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                images[0],
                scale=(0.7, 1.0),
                ratio=(0.85, 1.15),
            )
            do_flip = random.random() < 0.5
            brightness = 1.0 + random.uniform(-0.15, 0.15)
            contrast = 1.0 + random.uniform(-0.15, 0.15)
        else:
            resize_size = int(self.image_size * 1.15)

        for image in images:
            if self.train:
                image = TF.resized_crop(
                    image,
                    i,
                    j,
                    h,
                    w,
                    size=[self.image_size, self.image_size],
                    interpolation=InterpolationMode.BILINEAR,
                )
                if do_flip:
                    image = TF.hflip(image)
                image = TF.adjust_brightness(image, brightness)
                image = TF.adjust_contrast(image, contrast)
            else:
                image = TF.resize(
                    image,
                    size=[resize_size, resize_size],
                    interpolation=InterpolationMode.BILINEAR,
                )
                image = TF.center_crop(image, [self.image_size, self.image_size])

            tensor = TF.to_tensor(image)
            tensor = TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)
            processed.append(tensor)

        return torch.stack(processed, dim=0)


class MultiLabelVideoDataset(Dataset):
    def __init__(self, frame_df, label_cols, transform, num_frames):
        self.frame_df = frame_df.reset_index(drop=True)
        self.label_cols = label_cols
        self.transform = transform
        self.num_frames = num_frames

    def __len__(self):
        return len(self.frame_df)

    def __getitem__(self, idx):
        row = self.frame_df.iloc[idx]
        frame_dir = frame_dir_for(str(row["video_id"]))
        frame_paths = sample_frame_paths(frame_dir, self.num_frames)

        images = []
        for frame_path in frame_paths:
            with Image.open(frame_path) as img:
                images.append(img.convert("RGB"))

        clip = self.transform(images)
        labels = torch.tensor(row[self.label_cols].values, dtype=torch.float32)
        return clip, labels, str(row["video_id"])


train_transform = ClipTransform(image_size=CFG["image_size"], train=True)
val_transform = ClipTransform(image_size=CFG["image_size"], train=False)

train_dataset = MultiLabelVideoDataset(
    train_df,
    label_columns,
    train_transform,
    CFG["num_frames"],
)
val_dataset = MultiLabelVideoDataset(
    val_df,
    label_columns,
    val_transform,
    CFG["num_frames"],
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    # num_workers=CFG["num_workers"],
    pin_memory=USE_AMP,
    drop_last=False,
    # persistent_workers=CFG["num_workers"] > 0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    # num_workers=CFG["num_workers"],
    pin_memory=USE_AMP,
    drop_last=False,
    # persistent_workers=CFG["num_workers"] > 0,
)

sample_clip, sample_targets, sample_video_id = train_dataset[0]
print("sample video:", sample_video_id)
print("clip tensor:", tuple(sample_clip.shape))
print(
    "positive labels:",
    [
        label_columns[index]
        for index, value in enumerate(sample_targets.tolist())
        if value > 0
    ],
)


sample video: 2213
clip tensor: (8, 3, 224, 224)
positive labels: ['generic', 'sensitive', 'gender_hate']


In [12]:
# def load_efficientnet_v2_s():
#     if not hasattr(tv_models, "efficientnet_v2_s"):
#         raise AttributeError("efficientnet_v2_s is not available in this torchvision build")

#     builder = tv_models.efficientnet_v2_s
#     weights_enum = getattr(tv_models, "EfficientNet_V2_S_Weights", None)

#     try:
#         if weights_enum is not None:
#             model = builder(weights=weights_enum.DEFAULT)
#         else:
#             model = builder(pretrained=True)
#         pretrained = True
#     except Exception as exc:
#         print(f"EfficientNetV2-S pretrained weights unavailable: {exc}")
#         try:
#             model = builder(weights=None)
#         except TypeError:
#             model = builder(pretrained=False)
#         pretrained = False

#     feature_dim = model.classifier[1].in_features
#     model.classifier = nn.Identity()
#     return model, feature_dim, pretrained


# def load_resnet50():
#     builder = tv_models.resnet50
#     weights_enum = getattr(tv_models, "ResNet50_Weights", None)

#     try:
#         if weights_enum is not None:
#             model = builder(weights=weights_enum.DEFAULT)
#         else:
#             model = builder(pretrained=True)
#         pretrained = True
#     except Exception as exc:
#         print(f"ResNet50 pretrained weights unavailable: {exc}")
#         try:
#             model = builder(weights=None)
#         except TypeError:
#             model = builder(pretrained=False)
#         pretrained = False

#     feature_dim = model.fc.in_features
#     model.fc = nn.Identity()
#     return model, feature_dim, pretrained


# def build_backbone(preferred=None):
#     candidates = [preferred] if preferred else ["efficientnet_v2_s", "resnet50"]

#     for backbone_name in candidates:
#         if backbone_name == "efficientnet_v2_s":
#             try:
#                 backbone, feature_dim, pretrained = load_efficientnet_v2_s()
#                 return backbone_name, backbone, feature_dim, pretrained
#             except Exception as exc:
#                 print(f"Skipping efficientnet_v2_s: {exc}")
#         elif backbone_name == "resnet50":
#             backbone, feature_dim, pretrained = load_resnet50()
#             return backbone_name, backbone, feature_dim, pretrained
#         else:
#             raise ValueError(f"Unsupported backbone: {backbone_name}")

#     raise RuntimeError("Could not build a supported backbone")


# class VideoBaseline(nn.Module):
#     def __init__(self, num_labels, backbone_name=None):
#         super().__init__()
#         self.backbone_name, self.backbone, feature_dim, self.pretrained = build_backbone(
#             backbone_name
#         )
#         self.temporal_attention = nn.Sequential(
#             nn.LayerNorm(feature_dim),
#             nn.Linear(feature_dim, 256),
#             nn.GELU(),
#             nn.Dropout(0.1),
#             nn.Linear(256, 1),
#         )
#         self.head = nn.Sequential(
#             nn.LayerNorm(feature_dim),
#             nn.Dropout(0.35),
#             nn.Linear(feature_dim, num_labels),
#         )

#     def forward(self, clips):
#         batch_size, num_frames, channels, height, width = clips.shape
#         x = clips.view(batch_size * num_frames, channels, height, width)
#         features = self.backbone(x)
#         features = features.view(batch_size, num_frames, -1)
#         attn = torch.softmax(self.temporal_attention(features).squeeze(-1), dim=1)
#         pooled = (features * attn.unsqueeze(-1)).sum(dim=1)
#         return self.head(pooled)


# def set_backbone_trainable(model, trainable: bool):
#     for param in model.backbone.parameters():
#         param.requires_grad = trainable


def build_pos_weight(frame_df, label_cols, max_pos_weight):
    positives = frame_df[label_cols].sum().clip(lower=1)
    negatives = len(frame_df) - positives
    weights = (negatives / positives).clip(lower=1.0, upper=max_pos_weight)
    return torch.tensor(weights.values, dtype=torch.float32)


# model = VideoBaseline(len(label_columns)).to(DEVICE)
# set_backbone_trainable(model, False)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models


# ---------------------------
# Backbone loaders (unchanged)
# ---------------------------

def load_efficientnet_v2_s():
    builder = tv_models.efficientnet_v2_s
    weights_enum = getattr(tv_models, "EfficientNet_V2_S_Weights", None)

    try:
        model = builder(weights=weights_enum.DEFAULT) if weights_enum else builder(pretrained=True)
        pretrained = True
    except Exception:
        model = builder(weights=None) if weights_enum else builder(pretrained=False)
        pretrained = False

    feature_dim = model.classifier[1].in_features
    model.classifier = nn.Identity()
    return model, feature_dim, pretrained


def load_resnet50():
    builder = tv_models.resnet50
    weights_enum = getattr(tv_models, "ResNet50_Weights", None)

    try:
        model = builder(weights=weights_enum.DEFAULT) if weights_enum else builder(pretrained=True)
        pretrained = True
    except Exception:
        model = builder(weights=None) if weights_enum else builder(pretrained=False)
        pretrained = False

    feature_dim = model.fc.in_features
    model.fc = nn.Identity()
    return model, feature_dim, pretrained


def build_backbone(name=None):
    if name == "resnet50":
        return "resnet50", *load_resnet50()
    return "efficientnet_v2_s", *load_efficientnet_v2_s()


# ---------------------------
# Strong Video Model
# ---------------------------

class VideoModel(nn.Module):
    def __init__(self, num_labels, backbone_name=None):
        super().__init__()

        self.backbone_name, self.backbone, feature_dim, self.pretrained = build_backbone(backbone_name)

        # ---- Feature projection (critical) ----
        self.embed_dim = 512
        self.feature_proj = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, self.embed_dim),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.pos_embed = nn.Parameter(torch.randn(1, 500, self.embed_dim))
        

        # ---- CLS token ----
        self.cls_token = nn.Parameter(torch.randn(1, 1, self.embed_dim))

        # ---- Transformer temporal modeling ----
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim,
            nhead=8,
            dim_feedforward=1024,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )
        self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # ---- Classification head ----
        self.head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Linear(self.embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_labels),
        )

    def forward(self, clips):
        """
        clips: (B, T, C, H, W)
        """
        B, T, C, H, W = clips.shape

        # ---- frame-wise feature extraction ----
        x = clips.view(B * T, C, H, W)
        features = self.backbone(x)                # (B*T, F)
        features = features.view(B, T, -1)         # (B, T, F)

        # ---- project features ----
        features = self.feature_proj(features)     # (B, T, 512)

        # ---- add CLS token ----
        cls_tokens = self.cls_token.expand(B, -1, -1)
        features = torch.cat([cls_tokens, features], dim=1)  # (B, T+1, 512)
        # ---- temporal modeling ----
        features = features + self.pos_embed[:, :features.size(1)]
        features = self.temporal_encoder(features)

        # ---- classification using CLS ----
        pooled = features[:, 0]

        return self.head(pooled)


# ---------------------------
# Training utilities
# ---------------------------

def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.parameters():
        p.requires_grad = trainable


def unfreeze_last_layers(model, n=2):
    children = list(model.backbone.children())
    for layer in children[-n:]:
        for p in layer.parameters():
            p.requires_grad = True


# ---------------------------
# Optional: Focal Loss (better than BCE)
# ---------------------------

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        ce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')

        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_factor = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        modulating_factor = (1 - p_t) ** self.gamma

        loss = alpha_factor * modulating_factor * ce_loss
        return loss.mean()
    
    
model = VideoModel(len(label_columns)).to(DEVICE)
set_backbone_trainable(model, False)
unfreeze_last_layers(model, 2)

pos_weight = build_pos_weight(train_df, label_columns, CFG["max_pos_weight"]).to(DEVICE)
# criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion = FocalLoss()
scaler = GradScaler(enabled=USE_AMP)

print("backbone:", model.backbone_name, "| pretrained:", model.pretrained)
print(
    "trainable parameters:",
    f"{sum(param.numel() for param in model.parameters() if param.requires_grad):,}",
)
pd.Series(pos_weight.detach().cpu().numpy(), index=label_columns).sort_values(
    ascending=False
)




backbone: efficientnet_v2_s | pretrained: True
trainable parameters: 352,528


C:\Users\ASUS\AppData\Local\Temp\ipykernel_29292\689910848.py:115: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)


generic             20.000000
positive            20.000000
sexuality_hate      20.000000
indv_hate           20.000000
rare_hate           20.000000
informative         20.000000
nationality_hate    20.000000
gender_hate         15.193182
political_hate      12.701923
threat              10.875000
emotional            9.477942
derogatory__lang     4.089286
anger                3.781879
sensitive            1.553763
humour               1.000000
dtype: float32

In [ ]:
# ---- EMA ----
from copy import deepcopy

ema_decay = 0.999

def update_ema(model, ema_model):
    with torch.no_grad():
        for p, ema_p in zip(model.parameters(), ema_model.parameters()):
            ema_p.data.mul_(ema_decay).add_(p.data, alpha=1 - ema_decay)


NameError: name 'model' is not defined

In [ ]:
def build_optimizer(model, backbone_lr, head_lr):
    backbone_params = [param for param in model.backbone.parameters() if param.requires_grad]
    head_params = [
        param
        for module in (model.temporal_attention, model.head)
        for param in module.parameters()
        if param.requires_grad
    ]

    param_groups = []
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": backbone_lr})
    if head_params:
        param_groups.append({"params": head_params, "lr": head_lr})

    return torch.optim.AdamW(param_groups, weight_decay=CFG["weight_decay"])


def optimize_thresholds(y_true, y_prob, default=0.5):
    grid = np.linspace(0.15, 0.85, 15)
    thresholds = []

    for label_index in range(y_true.shape[1]):
        if y_true[:, label_index].sum() == 0:
            thresholds.append(default)
            continue

        best_threshold = default
        best_score = -1.0

        for threshold in grid:
            preds = (y_prob[:, label_index] >= threshold).astype(np.int32)
            score = f1_score(y_true[:, label_index], preds, zero_division=0)
            if score > best_score:
                best_score = score
                best_threshold = float(threshold)

        thresholds.append(best_threshold)

    return np.array(thresholds, dtype=np.float32)


def compute_metrics(y_true, y_prob, thresholds):
    thresholds = np.asarray(thresholds, dtype=np.float32)
    preds = (y_prob >= thresholds.reshape(1, -1)).astype(np.int32)

    per_label_f1 = f1_score(y_true, preds, average=None, zero_division=0)
    macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, preds, average="micro", zero_division=0)
    hamming_acc = (preds == y_true).mean()
    subset_acc = (preds == y_true).all(axis=1).mean()
    label_acc = hamming_acc
    exact_match_acc = subset_acc

    ap_scores = []
    for label_index in range(y_true.shape[1]):
        if len(np.unique(y_true[:, label_index])) < 2:
            ap_scores.append(np.nan)
        else:
            ap_scores.append(
                average_precision_score(y_true[:, label_index], y_prob[:, label_index])
            )

    ap_scores = np.asarray(ap_scores, dtype=np.float32)
    macro_ap = float(np.nanmean(ap_scores)) if not np.isnan(ap_scores).all() else float("nan")

    return {
        "macro_f1": float(macro_f1),
        "micro_f1": float(micro_f1),
        "macro_ap": macro_ap,
        "hamming_acc": float(hamming_acc),
        "subset_acc": float(subset_acc),
        "label_acc": float(label_acc),
        "exact_match_acc": float(exact_match_acc),
        "per_label_f1": per_label_f1,
        "preds": preds,
    }


def run_epoch(model, loader, criterion, ema_model=None, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_probs = []
    all_targets = []

    progress = tqdm(loader, leave=False)
    progress.set_description("train" if is_train else "val")

    grad_context = torch.enable_grad() if is_train else torch.no_grad()
    amp_enabled = bool(scaler is not None and scaler.is_enabled())

    with grad_context:
        for clips, labels, _ in progress:
            clips = clips.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=DEVICE.type, enabled=amp_enabled):
                logits = model(clips)
                loss = criterion(logits, labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                
            if ema_model is not None:
                update_ema(model, ema_model)

            total_loss += loss.item() * clips.size(0)
            all_probs.append(torch.sigmoid(logits).detach().cpu())
            all_targets.append(labels.detach().cpu())
            progress.set_postfix(loss=f"{loss.item():.4f}")

    y_prob = torch.cat(all_probs).numpy()
    y_true = torch.cat(all_targets).numpy().astype(np.int32)
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, y_true, y_prob


def save_checkpoint(path, model, thresholds, history):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "label_columns": label_columns,
            "config": CFG,
            "thresholds": thresholds,
            "history": history,
            "backbone": model.backbone_name,
        },
        path,
    )


In [ ]:
optimizer = build_optimizer(model, CFG["backbone_lr"], CFG["head_lr"])
scheduler = None
ema_model = deepcopy(model)

best_macro_f1 = -1.0
best_thresholds = np.full(len(label_columns), 0.5, dtype=np.float32)
best_val_y = None
best_val_prob = None
history = []
epochs_without_improvement = 0

for epoch in range(1, CFG["epochs"] + 1):
    if epoch == CFG["warmup_epochs"] + 1:
        set_backbone_trainable(model, True)
        optimizer = build_optimizer(model, CFG["backbone_lr"], CFG["head_lr"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(1, CFG["epochs"] - CFG["warmup_epochs"]),
        )
        print(f"epoch {epoch:02d}: backbone unfrozen")

    train_loss, train_y, train_prob = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
        scaler=scaler,
        ema_model=ema_model
    )
    val_loss, val_y, val_prob = run_epoch(model, val_loader, criterion,ema_model)

    thresholds = optimize_thresholds(val_y, val_prob, default=0.5)
    train_metrics = compute_metrics(train_y, train_prob, thresholds)
    val_metrics = compute_metrics(val_y, val_prob, thresholds)

    if scheduler is not None:
        scheduler.step()

    epoch_row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_macro_f1": train_metrics["macro_f1"],
        "val_macro_f1": val_metrics["macro_f1"],
        "train_label_acc": train_metrics["label_acc"],
        "val_label_acc": val_metrics["label_acc"],
        "train_exact_match_acc": train_metrics["exact_match_acc"],
        "val_exact_match_acc": val_metrics["exact_match_acc"],
        "val_micro_f1": val_metrics["micro_f1"],
        "val_macro_ap": val_metrics["macro_ap"],
    }
    history.append(epoch_row)

    print(
        f"epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"train_acc={train_metrics['label_acc']:.4f} | "
        f"val_acc={val_metrics['label_acc']:.4f} | "
        f"train_exact_acc={train_metrics['exact_match_acc']:.4f} | "
        f"val_exact_acc={val_metrics['exact_match_acc']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
        f"val_micro_f1={val_metrics['micro_f1']:.4f} | "
        f"val_macro_ap={val_metrics['macro_ap']:.4f}"
    )

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_thresholds = thresholds.copy()
        best_val_y = val_y.copy()
        best_val_prob = val_prob.copy()
        epochs_without_improvement = 0
        save_checkpoint(CHECKPOINT_PATH, model, best_thresholds, history)
        save_checkpoint(CHECKPOINT_PATH_EMA, ema_model, best_thresholds, history)
        print(f"saved best checkpoint -> {CHECKPOINT_PATH.name}")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CFG["patience"]:
            print("early stopping triggered")
            break

history_df = pd.DataFrame(history)
display(history_df)

best_val_metrics = compute_metrics(best_val_y, best_val_prob, best_thresholds)
per_label_df = pd.DataFrame(
    {
        "label": label_columns,
        "threshold": best_thresholds,
        "val_f1": best_val_metrics["per_label_f1"],
        "train_positives": train_df[label_columns].sum().values,
        "val_positives": val_df[label_columns].sum().values,
    }
).sort_values("val_f1", ascending=False)

display(per_label_df)


  0%|          | 0/179 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

epoch 01 | train_loss=1.1012 | val_loss=1.0171 | train_acc=0.6609 | val_acc=0.7294 | train_exact_acc=0.0014 | val_exact_acc=0.0084 | val_macro_f1=0.2779 | val_micro_f1=0.4463 | val_macro_ap=0.2031
saved best checkpoint -> best_model_fast_baseline.pt
epoch 02: backbone unfrozen


  0%|          | 0/179 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

...

  0%|          | 0/179 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

epoch 04 | train_loss=0.7987 | val_loss=0.9675 | train_acc=0.8195 | val_acc=0.8105 | train_exact_acc=0.0512 | val_exact_acc=0.0672 | val_macro_f1=0.3393 | val_micro_f1=0.5250 | val_macro_ap=0.2772


  0%|          | 0/179 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def load_checkpoint(checkpoint_path=CHECKPOINT_PATH, device=DEVICE):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = VideoBaseline(
        len(checkpoint["label_columns"]),
        backbone_name=checkpoint.get("backbone"),
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model, checkpoint


def predict_from_frame_dir(frame_dir, checkpoint_path=CHECKPOINT_PATH):
    model, checkpoint = load_checkpoint(checkpoint_path)
    frame_dir = Path(frame_dir)
    frame_paths = sample_frame_paths(frame_dir, checkpoint["config"]["num_frames"])

    images = []
    for frame_path in frame_paths:
        with Image.open(frame_path) as img:
            images.append(img.convert("RGB"))

    inference_transform = ClipTransform(
        image_size=checkpoint["config"]["image_size"],
        train=False,
    )
    clip = inference_transform(images).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.sigmoid(model(clip)).squeeze(0).cpu().numpy()

    thresholds = np.asarray(checkpoint["thresholds"], dtype=np.float32)
    preds = (probs >= thresholds).astype(int)

    return pd.DataFrame(
        {
            "label": checkpoint["label_columns"],
            "prob": probs,
            "threshold": thresholds,
            "pred": preds,
        }
    ).sort_values("prob", ascending=False)


# Example usage after training:
# predict_from_frame_dir(FRAMES_ROOT / Path(train_df.loc[0, "video_id"]).stem)
